# Module 09 — The Data Model: Dunder Methods

## Exercise 09.4 — Four context managers, two ways each

Write each as a class with __enter__/__exit__, then again with
@contextlib.contextmanager, and note which you would ship.
Run:  python ex04_context.py

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.

---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 5. Iteration

In [ ]:
class Countdown:
    def __init__(self, start: int) -> None:
        self.start = start

    def __iter__(self):
        return CountdownIterator(self.start)      # a FRESH iterator each time


class CountdownIterator:
    def __init__(self, current: int) -> None:
        self.current = current

    def __iter__(self):
        return self                                # iterators return themselves

    def __next__(self) -> int:
        if self.current <= 0:
            raise StopIteration                    # the protocol's "done"
        self.current -= 1
        return self.current + 1

**Iterable versus iterator** is the distinction that matters:

| | Iterable | Iterator |
|---|---|---|
| Defines | `__iter__` | `__iter__` **and** `__next__` |
| Reusable | Yes — a fresh iterator each time | **No.** Once exhausted, done. |
| Examples | `list`, `dict`, `str`, `range` | `iter([])`, a generator, a file object |

Making a class its own iterator (returning `self` from `__iter__` and keeping the
position on the instance) means **two `for` loops over it cannot both work** —
the second sees an exhausted object. That is a real and confusing bug. Return a
fresh iterator, or write `__iter__` as a generator, which does it for you:

```text
    def __iter__(self):
        current = self.start          # local state -> fresh each call
        while current > 0:
            yield current
            current -= 1
```


Module 14 covers generators properly.

---

## Concept 6. Context managers

In [ ]:
class Timer:
    def __enter__(self) -> "Timer":
        self.start = time.perf_counter()
        return self                    # what `as x` binds

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        self.elapsed = time.perf_counter() - self.start
        return False                   # False/None: do NOT suppress exceptions

`__exit__` runs **whether or not** an exception occurred — that is the entire
point. Its three arguments are `None, None, None` on a clean exit.

**Returning `True` from `__exit__` swallows the exception.** Almost always
wrong. Do it only when suppression is the explicit purpose, as in
`contextlib.suppress`.

The concise form, which is what you will actually write (Module 15):

In [ ]:
from contextlib import contextmanager

@contextmanager
def timer():
    start = time.perf_counter()
    try:
        yield
    finally:                      # finally, not bare -- runs on exception too
        print(f"{time.perf_counter() - start:.3f}s")

---

## Concept 7. Operators

In [ ]:
class Vector:
    def __init__(self, x: float, y: float) -> None:
        self.x, self.y = x, y

    def __add__(self, other: "Vector") -> "Vector":
        if not isinstance(other, Vector):
            return NotImplemented
        return Vector(self.x + other.x, self.y + other.y)

    def __mul__(self, scalar: float) -> "Vector":
        if not isinstance(scalar, (int, float)):
            return NotImplemented
        return Vector(self.x * scalar, self.y * scalar)

    __rmul__ = __mul__            # makes 3 * v work as well as v * 3

    def __neg__(self) -> "Vector":
        return Vector(-self.x, -self.y)

    def __abs__(self) -> float:
        return (self.x**2 + self.y**2) ** 0.5

**How Python resolves `a + b`:**

1. Try `type(a).__add__(a, b)`. If it returns `NotImplemented`, continue.
2. Try `type(b).__radd__(b, a)`. If that also returns `NotImplemented`:
3. `TypeError: unsupported operand type(s)`.

(With one refinement: if `type(b)` is a *subclass* of `type(a)`, the reflected
method is tried first, so a subclass can override its parent's behaviour.)

This is why `NotImplemented` matters. Returning it is how you say "not my
problem" and let the other operand try. Note the trap: `NotImplemented` is
**truthy**, so accidentally returning it from `__eq__` and using the result in an
`if` gives you a silent wrong answer plus a `DeprecationWarning`.

**In-place operators** (`__iadd__` etc.) should mutate and `return self` — for a
mutable type. For an immutable one, omit them and Python falls back to
`__add__` plus rebinding. This is exactly Module 02's list-versus-tuple `+=`
distinction, now from the implementer's side.

Only overload operators where the meaning is obvious. `Vector + Vector` is
clear. `User + User` is not, and a `merge()` method would be better.

---

## Concept 9. The whole map

| Group | Methods |
|---|---|
| Representation | `__repr__` `__str__` `__format__` `__bytes__` |
| Comparison | `__eq__` `__ne__` `__lt__` `__le__` `__gt__` `__ge__` `__hash__` |
| Container | `__len__` `__getitem__` `__setitem__` `__delitem__` `__contains__` `__reversed__` |
| Iteration | `__iter__` `__next__` `__aiter__` `__anext__` |
| Numeric | `__add__` `__sub__` `__mul__` `__truediv__` `__floordiv__` `__mod__` `__pow__` `__neg__` `__abs__` `__round__` and the `__r*__` / `__i*__` variants |
| Conversion | `__bool__` `__int__` `__float__` `__index__` `__complex__` |
| Context | `__enter__` `__exit__` `__aenter__` `__aexit__` |
| Callable | `__call__` |
| Attributes | `__getattr__` `__getattribute__` `__setattr__` `__delattr__` `__dir__` |
| Descriptors | `__get__` `__set__` `__delete__` `__set_name__` |
| Class machinery | `__init__` `__new__` `__init_subclass__` `__class_getitem__` `__slots__` |
| Copying | `__copy__` `__deepcopy__` `__reduce__` |
| Pattern matching | `__match_args__` |

You do not need to memorise this. You need to know it exists, so that when you
want your type to work with some piece of syntax, you look up which method
provides it.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: `__repr__` and `__str__`
- Section 2: `__eq__` and `__hash__` are a pair
- Section 3: Ordering
- Section 4: The container protocols
- Section 5: Iteration
- Section 6: Context managers
- Section 7: Operators
- Section 8: `__call__`, `__bool__`, `__format__`
- Section 9: The whole map

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

import time
from contextlib import contextmanager
from pathlib import Path
from typing import Any


# TODO 1 -----------------------------------------------------------------------

---

## `Timer`

Measures the wall time of a block.

In [ ]:
class Timer:
    """Measures the wall time of a block.

    Requirements:
      - `with Timer() as t:` then t.elapsed afterwards
      - must record the time EVEN IF the block raises (that is the whole point
        of __exit__ over a plain function call)
      - must not swallow the exception
      - optional label, printed on exit only if one was given
      - use time.perf_counter, not time.time. Say why in a comment.
    """

---

## `timer`

The same thing with @contextmanager. Note where the try/finally goes and

In [ ]:
@contextmanager
def timer(label: str = ""):  # type: ignore[no-untyped-def]
    """The same thing with @contextmanager. Note where the try/finally goes and
    why a bare try/except would be wrong."""
    raise NotImplementedError

---

## `Transaction`

A fake transaction with commit and rollback.

In [ ]:
class Transaction:
    """A fake transaction with commit and rollback.

    Requirements:
      - on clean exit: commit
      - on exception:  rollback, then let the exception propagate
      - nested transactions become SAVEPOINTS: an inner rollback must not
        discard the outer transaction's work
      - .operations records what happened, in order, so tests can assert on it

    Then answer in a comment: why must __exit__ re-raise rather than return
    True, even though "the rollback handled it" sounds reasonable?
    """

    def __init__(self, name: str = "tx") -> None:
        self.name = name
        self.operations: list[str] = []

---

## `temporary_directory`

Create a temp directory, yield its Path, remove it afterwards.

In [ ]:
@contextmanager
def temporary_directory(prefix: str = "tmp"):  # type: ignore[no-untyped-def]
    """Create a temp directory, yield its Path, remove it afterwards.

    Requirements:
      - removal happens even if the block raises
      - removal happens even if the block CREATED files inside it
      - if removal itself fails, do not mask the block's exception

    That last requirement is subtle and is where most hand-rolled cleanup goes
    wrong. Work out what happens if both the body and the cleanup raise, and
    write down which exception a caller sees and which is lost.
    """
    raise NotImplementedError

---

## `suppress_and_log`

Like contextlib.suppress, but records what it swallowed.

In [ ]:
class suppress_and_log:
    """Like contextlib.suppress, but records what it swallowed.

    Requirements:
      - suppress ONLY the listed exception types
      - record each suppressed exception in .caught
      - anything not listed propagates untouched
      - be reusable and REENTRANT: the same instance used in two nested with
        blocks must work

    Then answer: contextlib.suppress is one of the very few legitimate uses of
    returning True from __exit__. What makes it legitimate, and what makes
    almost every other use of it a bug?
    """

    def __init__(self, *exceptions: type[BaseException]) -> None:
        self.exceptions = exceptions
        self.caught: list[BaseException] = []

---

## `verify`

_verify_

In [ ]:
def verify() -> None:
    with Timer() as t:
        time.sleep(0.01)
    assert t.elapsed >= 0.01

    try:
        with Timer() as t2:
            raise ValueError("boom")
    except ValueError:
        pass
    assert t2.elapsed > 0, "timer must record even when the block raises"

    tx = Transaction()
    with tx:
        tx.operations.append("insert")
    assert tx.operations[-1] == "commit", tx.operations

    tx2 = Transaction()
    try:
        with tx2:
            tx2.operations.append("insert")
            raise RuntimeError("fail")
    except RuntimeError:
        pass
    else:
        raise AssertionError("exception must propagate")
    assert tx2.operations[-1] == "rollback", tx2.operations

    with temporary_directory() as d:
        assert d.is_dir()
        (d / "file.txt").write_text("data", encoding="utf-8")
        kept = d
    assert not kept.exists(), "temp directory survived"

    try:
        with temporary_directory() as d2:
            kept2 = d2
            raise ValueError("boom")
    except ValueError:
        pass
    assert not kept2.exists(), "temp directory survived an exception"

    s = suppress_and_log(ValueError, KeyError)
    with s:
        raise ValueError("swallowed")
    assert len(s.caught) == 1

    with s:
        raise KeyError("also swallowed")
    assert len(s.caught) == 2

    try:
        with s:
            raise TypeError("not listed")
    except TypeError:
        pass
    else:
        raise AssertionError("unlisted exceptions must propagate")

    print("all context manager checks passed")

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    verify()

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.